#  Import required packages


In [40]:
import pandas as pd
from pathlib import Path
import numpy as np
import os
import yaml
%matplotlib inline
%config InlineBackend.figure_format='retina'

# Required Inputs


In [41]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Set up directories
raw_dir = Path(config['raw_dir'])
metadata_dir = Path(config['metadata_dir'])
preprocessed_dir = Path(config['preprocessed_dir'])

# make a roi list of directories in raw_dir that are not .DS_Store
roi_list = [roi for roi in raw_dir.iterdir() if roi.is_dir() and roi.name != '.DS_Store']

#make antibody panel list
ab_panels = [ab for ab in metadata_dir.iterdir() if ab.is_file() and 'Ab_panel.xlsx' in ab.name]

# Select the phenotypes depth to take from the MACSIMA output
pheno_depth=int(3)

# Preparing the cell phenotypes dataframe

## 1) Extract phenotype lineage information

In [43]:
lineage_info={}

lineage_matrix = pd.read_csv(metadata_dir/"expression_table_sophia.csv", sep=';')

for cell_type,lineage in zip(lineage_matrix.iloc[:,0].values,lineage_matrix.iloc[:,1].values):

    lineage_info[cell_type]=tuple(np.array(lineage.split('_')).astype('int'))

lineage_info

{'T_cells': (1, 0, 1),
 'CD4+_T_cells': (2, 1, 2),
 'CD8+_T_cells': (2, 1, 3),
 'Treg_T_cells': (3, 2, 4),
 'B_cells': (1, 0, 5),
 'Plasma_cells': (2, 5, 6),
 'Granulocytes': (1, 0, 7),
 'NK_cells': (1, 0, 8),
 'Mast_cells': (1, 0, 9),
 'Macrophages': (1, 0, 10),
 'M1_macrophages': (2, 10, 11),
 'M2_macrophages': (2, 10, 12),
 'M1_M2_macrophages': (2, 10, 13),
 'Tumor_cells': (1, 0, 14),
 'Endothelial_cells': (1, 0, 15),
 'Lymphatic_endothelial_cells': (1, 0, 16),
 'Actin+_cells': (1, 0, 17),
 'DCs': (1, 0, 18),
 'Epithelial_cells': (1, 0, 19),
 'MDSCs': (1, 0, 20),
 'NFC': (1, 0, 21),
 'Trash': (1, 0, 22)}

## 2) Concatenate all cell type files into a single file and attach lineage info

In [44]:
def concat_MACSiq(work_dir,preprocessed_dir=preprocessed_dir):
    files = [x for x in work_dir.iterdir() if x.is_file() and x.suffix == '.csv' and x.name != '1 all_info.csv']
    output_dir = Path(preprocessed_dir/work_dir.name)
    if os.path.exists(output_dir):
        pass 
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
    output_file = Path(output_dir / 'macsiq_phenotyping.csv')
    if os.path.exists(output_file):
        print("File exists")
        return pd.read_csv(output_file)
    else:
        pass
    data=[]
    for f in files:
        df=pd.read_csv(f)
        symbol='@'+df['Cell Id'][0].split('@')[1]
        df.insert(1,'cell_type',df.shape[0]*[f.name[0:-4]])
        df.insert(1,'cell_id',''.join(df['Cell Id'].values).split(symbol)[0:-1])
        data.append(df)
    
    df=pd.concat(data,ignore_index=True)
    df["cell_id"] = pd.to_numeric(df["cell_id"])
    df.sort_values(by=['cell_id'],inplace=True,ignore_index=True)
    df_macsiq=df.loc[:,['cell_id','cell_type']]
    df_macsiq['cell_type'] = df_macsiq['cell_type'].str.replace(r'^\d+\s+', '', regex=True)

    level=[]
    source=[]
    cell_type_label=[]
    for c in df_macsiq.cell_type.values:
        info=lineage_info[c]
        level.append(info[0])
        source.append(info[1])
        cell_type_label.append(info[2])

    df_lineage=pd.DataFrame({'level':level,'parent_cell':source,'type_no': cell_type_label })
    df_macsiq=pd.concat([df_macsiq,df_lineage],ignore_index=False,axis=1)

    df_macsiq.to_csv(output_file,index=False)

    return df_macsiq

for work_dir in roi_list:
    concat_MACSiq(work_dir, preprocessed_dir)

## 3) Filter out unwanted values: A) phenotyping depths/levels and B) repeated labels such that only the highest phenotyping level/depth remains

In [45]:
def filter_MACSiq(work_dir, preprocessed_dir=preprocessed_dir):
    output_dir = Path(preprocessed_dir/work_dir.name)
    input_file = Path(output_dir / 'macsiq_phenotyping.csv')
    output_file = Path(output_dir/'final_phenotyping_labels.csv')
    if os.path.exists(output_file):
        print(f"{output_file.stem} already exists")
        return pd.read_csv(output_file)
    else:
        pass
    print(f"Using {input_file.stem} to create {output_file.stem}")
    df_macsiq = pd.read_csv(input_file)
    df_macsiq_filt = df_macsiq.loc[df_macsiq['level']<=pheno_depth]
    repeated_labels=[cellID for cellID, rep in (df_macsiq_filt['cell_id'].value_counts()>1).items() if rep==True]
    repeated_labels.sort()
    remove_indices=[]
    for cellID in repeated_labels:
        cell_lineage=df_macsiq_filt.loc[df_macsiq_filt.cell_id==cellID].level
        index=np.argmax(cell_lineage.values)
        final_type_index=cell_lineage.index.values[index]
        remove_elements=np.setdiff1d(cell_lineage.index.values,[final_type_index]).tolist()
        if remove_elements:
            remove_indices.extend(remove_elements)
    df_macsiq_filt=df_macsiq_filt.drop(index=remove_indices)
    df_macsiq_filt.to_csv(output_file,index=False)
    return df_macsiq_filt

for work_dir in roi_list:
    filter_MACSiq(work_dir, preprocessed_dir)

Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final_phenotyping_labels
Using macsiq_phenotyping to create final

# Preparing the expression matrix and the metadata

## 4) Extract each roi's cells metadata

In [46]:
def meta_MACSiq(work_dir, preprocessed_dir=preprocessed_dir):
    output_dir = Path(preprocessed_dir/work_dir.name)
    output_file = Path(output_dir / 'metadata.csv')
    if os.path.exists(output_file):
        print(f"{output_file.stem} already exists")
        return pd.read_csv(output_file)
    else:
        pass
    print(f"Creating metadata file {output_file.stem}") 
    #load the data with the metadata columns only (79 columns)
    meta = pd.read_csv(work_dir / '1 all_info.csv', usecols=range(1,79))
    #add cell_id column
    meta.insert(0, "cell_id", range(1, 1 + len(meta)))
    #loading the phenotyping data
    df_macsiq_filt = pd.read_csv(output_dir/'final_phenotyping_labels.csv')
    #merge the metadata with the phenotyping data
    meta_joined = df_macsiq_filt.merge(meta, how="left")
    #export the metadata file
    meta_joined.to_csv(output_file,index=False)
    return meta_joined

for work_dir in roi_list:
    meta_MACSiq(work_dir, preprocessed_dir)

Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata
Creating metadata file metadata


## 5) Prepare the antibody panel data

In [47]:
#Read the antibodies list
def fix_antibody(panel):
    #check which panel is being used
    if panel.name == "Run92_Ab_panel.xlsx":
        ab = pd.read_excel(panel, sheet_name='IO Cryo Panel')
    else:
        ab = pd.read_excel(panel, sheet_name=["IO core panel", "Add-on Panel"])
        ab = pd.concat(ab, ignore_index=True)
    
    ab["antibody_name"] = ab.apply(lambda x: x["ab_clone"].split(f'{x["clone"]}')[0], axis=1)
    ab["antibody_name"] = ab["antibody_name"].str.replace("__", "_").str.replace("_", "").str.replace("-", "").str.replace(" ", "").sort_values()
    #remove antibodies with low staining quality
    ab = ab[ab["staining quality"] != "LOW"]
    #add 1 at the end of antibody name if it is duplicated
    ab["antibody_name"][ab["antibody_name"].duplicated(keep="first")] = ab["antibody_name"][ab["antibody_name"].duplicated(keep="first")] + " 1"
    ab["antibody_name"] = ab["antibody_name"].str.replace("Arginase12", "Arginase1")
    ab["antibody_name"] = ab["antibody_name"].str.replace("CollagenIV2", "CollagenIV")
    ab["antibody_name"] = ab["antibody_name"].str.replace("S1001", "S100")
    ab["antibody_name"] = ab["antibody_name"].str.upper()
    ab["exp_col"] = ab["ab_clone"].str.replace("__", " ").str.replace("_", " ")
    ab["exp_col"][ab["exp_col"].duplicated(keep="first")] = ab["exp_col"][ab["exp_col"].duplicated(keep="first")] + " 1"
    #sort them by exp col
    ab = ab.sort_values("exp_col")
    #export it to csv
    ab.to_csv(metadata_dir/f"{panel.stem}.csv", index=False)
    return ab

ab_list = []
for panel in ab_panels:
    ab = fix_antibody(panel)
    ab_list.append(ab["antibody_name"].values)
    # ab_intersection = set(ab_list[0]).intersection(*ab_list)

/var/folders/zj/m3tnl76s7d5894m0ndctg1kh0000gn/T/ipykernel_1059/3256330975.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  ab["antibody_name"][ab["antibody_name"].duplicated(keep="first")] = ab["antibody_name"][ab["antibody_name"].dupli

## 6) Extract expression data 

In [48]:
def exp_MACSiq(work_dir, preprocessed_dir=preprocessed_dir):
    output_dir = Path(preprocessed_dir/work_dir.name)
    output_file = Path(output_dir / 'exp.csv')
    run = work_dir.name.split('_')[0]
    if os.path.exists(output_file):
        print(f"{output_file.stem} already exists")
        return pd.read_csv(output_file)
    else:
        pass
    print("Creating expression file", output_file)
    #load the necessary antibody list
    ab_exp = pd.read_csv(metadata_dir/f"{run}_Ab_panel.csv")
    #load the expression data
    all_columns = pd.read_csv(work_dir / '1 all_info.csv', nrows=0).columns
    selected_columns = all_columns[all_columns.str.contains("Cell Exp")]
    exp = pd.read_csv(work_dir / '1 all_info.csv', usecols=selected_columns)
    #remove ' Cell Exp' from column names
    exp.columns = exp.columns.str.replace(' Cell Exp', "")
    #Take out CD279 REA1165 1 if available
    exp = exp.drop(columns="CD279 REA1165 1", errors="ignore")
    #Filter out the columns for only ab_exp columns
    available_cols = list(set(exp.columns) & set(ab_exp["exp_col"]))
    ab_exp = ab_exp[ab_exp["exp_col"].isin(available_cols)]
    exp = exp.loc[:,available_cols]
    #sort columns alphabetically
    exp = exp.reindex(sorted(exp.columns), axis=1)
    #Fix the names of the antibodies
    exp.columns = ab_exp["antibody_name"].values
    #add cell_id column
    exp.insert(0, 'cell_id', range(1, 1 + len(exp)))
    #Only keep rows with a cell type (Remove trash cells)
    meta = pd.read_csv(output_dir / 'metadata.csv')
    cell_ids = meta['cell_id']
    exp = exp[exp['cell_id'].isin(cell_ids)]
    #export the expression data
    exp.to_csv(output_file,index=False)
    return exp
 
for work_dir in roi_list:
    exp_MACSiq(work_dir, preprocessed_dir)

Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI11/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI16/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI7/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run145_ROI3/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI19/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI1/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI8/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run141_ROI17/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run145_ROI10/exp.csv
Creating expression file /Users/jawadalaaedeen/Desktop/PhD/NMC/preprocessed/Run145_ROI17/exp.csv
Creating expression file /Users/ja

## 7) Extract patients metadata and export it to the new format

In [49]:
patients_metadata = pd.read_excel(metadata_dir / 'patient_metadata.xlsx')
patients_metadata["outcome"] = patients_metadata["outcome"].str.extract('(\d+)').astype(int)
#remove excluded rois
patients_metadata = patients_metadata.loc[~patients_metadata["rois"].isna()]
#add a new column if the  "outcome" column is greater than 9
patients_metadata["outcome_group"] = np.where(patients_metadata["outcome"] > 6, "long_term survival", "short_term survival")
patients_metadata["rois"] = patients_metadata["rois"].astype(str)
patients_metadata['rois'] = patients_metadata['rois'].str.split(r'\+ ')
patients_metadata = patients_metadata.explode('rois', ignore_index=True)
patients_metadata['rois'] = patients_metadata['rois'].str.strip()
for i in range(len(patients_metadata)):
    if patients_metadata['rois'][i].find('-') != -1:
        new_patient = patients_metadata['rois'][i].split('-')
        new_patient = np.array(range(int(new_patient[0]), int(new_patient[1])+1))
        patients_metadata['rois'][i] = new_patient

patients_metadata = patients_metadata.explode('rois', ignore_index=True)
patients_metadata["rois"] = patients_metadata["rois"].astype(str)
patients_metadata["exp_name"] = "Run" +  patients_metadata['run'].astype(int).astype(str) + '_ROI' + patients_metadata['rois'].astype(str)
patients_metadata.to_csv(metadata_dir / 'new_patients_metadata.csv', index=False)

/var/folders/zj/m3tnl76s7d5894m0ndctg1kh0000gn/T/ipykernel_1059/757644536.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  patients_metadata['rois'][i] = new_patient
/var/folders/zj/m3tnl76s7d5894m0ndctg1kh0000gn/T/ipykernel_1059/7576445

## 8) Remove the trash cells

In [53]:
processed_roi_list = [roi for roi in preprocessed_dir.iterdir() if roi.is_dir() and roi.name != '.DS_Store']
for roi in processed_roi_list:
    exp = pd.read_csv(roi / 'exp.csv')
    meta = pd.read_csv(roi / 'metadata.csv')
    if "Trash" not in meta["cell_type"].unique():
        print("No trash cells in metadata")
        continue
    else:
        pass
    print("Removing trash cells")
    meta = meta.loc[meta["cell_type"] != "Trash"]
    exp = exp[exp["cell_id"].isin(meta.cell_id)]
    #export csvs
    exp.to_csv(roi / 'exp.csv', index=False)
    meta.to_csv(roi / 'metadata.csv', index=False)

No trash cells in metadata
No trash cells in metadata
Removing trash cells
Removing trash cells
No trash cells in metadata
No trash cells in metadata
Removing trash cells
Removing trash cells
Removing trash cells
Removing trash cells
Removing trash cells
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
Removing trash cells
No trash cells in metadata
No trash cells in metadata
No trash cells in metadata
Removing trash cells
No trash cells in metadata
Removing trash cells
Removing trash cells
No trash cells in metadata
No trash cells in metadata


## 9) Quick QC : Extract the names of all cell types

In [54]:
# Create a list of all cell types
all_cell_types = []  
for i, roi in enumerate(processed_roi_list):
    meta = pd.read_csv(roi / 'metadata.csv')
    all_cell_types.extend(meta["cell_type"].unique())
# Create a set of unique cell types
all_cell_types = sorted(set(all_cell_types))   
# Create a DataFrame with all cell types and initialize counts to zero
meta = pd.read_csv(processed_roi_list[0] / 'metadata.csv')
meta_counts = meta["cell_type"].value_counts()
for cell_type in all_cell_types:
        if cell_type not in meta_counts:
            meta_counts[cell_type] = 0
sorted_order = meta_counts.index  # Extract the index order from a reference DataFrame
all_cell_types = list(sorted_order)
all_cell_types

['Tumor_cells',
 'Actin+_cells',
 'NFC',
 'Endothelial_cells',
 'CD4+_T_cells',
 'M1_macrophages',
 'DCs',
 'M1_M2_macrophages',
 'Mast_cells',
 'M2_macrophages',
 'CD8+_T_cells',
 'Treg_T_cells',
 'Plasma_cells',
 'Lymphatic_endothelial_cells',
 'MDSCs',
 'B_cells',
 'Granulocytes',
 'Epithelial_cells',
 'NK_cells']

## 10) Quick QC : Checking  the number of antibodies in all samples and runs

In [55]:
for roi in processed_roi_list:
    exp = pd.read_csv(roi / 'exp.csv')
    print (roi.name)
    print(len(exp.columns))

Run141_ROI11
87
Run141_ROI16
87
Run141_ROI7
87
Run145_ROI3
74
Run141_ROI19
87
Run141_ROI1
87
Run141_ROI8
87
Run141_ROI17
87
Run145_ROI10
74
Run145_ROI17
71
Run145_ROI11
74
Run103_ROI5
61
Run92_ROI5
53
Run141_ROI15
87
Run145_ROI1
74
Run141_ROI4
87
Run141_ROI23
87
Run141_ROI3
87
Run145_ROI9
74
Run141_ROI14
87
Run145_ROI14
74
Run145_ROI13
74
Run103_ROI6
61
Run92_ROI6
53
